# Bag of words, TF-IDF en een eerste aanbeveling op basis van kenmerken

Deze notebook leert om een tekst om te zetten in getallen en eindigt met een klein aanbevelingssysteem: voor elke omschrijving de vijf meest gelijkende omschrijvingen.

## 1. Voorbereiding

Nodig: `pandas`, `scikit-learn` en `nltk` (voor de Nederlandse stopwoordenlijst en de stemmer). Alle drie zijn verplicht: zonder `nltk` geeft de notebook een foutmelding. Installeer ze met de cel hieronder.

In [99]:
# Installeert de pakketten die deze notebook nodig heeft. Een pakket dat al aanwezig is, wordt overgeslagen.
%pip install -q pandas scikit-learn nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [100]:
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 2. Wat is hier een document?

Alles hierna werkt op een **lijst van teksten**. Elk element van die lijst is één *document* en wordt straks één rij in de matrix. In de catalogus is dat makkelijk: één rij in de csv is één omschrijving. Zinnen splitsen hoeft dus niet; een omschrijving van drie zinnen is gewoon één document.

Eerst de drie zinnen van het bord. Let op: het is een lijst met drie strings, niet één lange string.

In [101]:
demo_sentences = [
    "De hond bijt de man",
    "De kat slaapt",
    "De hond slaapt, de kat slaapt",
]
for i, sentence in enumerate(demo_sentences, 1):
    print(i, sentence)

1 De hond bijt de man
2 De kat slaapt
3 De hond slaapt, de kat slaapt


In [102]:
books = pd.DataFrame({
    "Titel": [
        "De draak onder de school", "Leven in de ruimte", "Het drakenei", "Nieuw in de stad",
        "Code rood op de redactie", "Vulkanen van dichtbij", "De laatste wolf", "Mijn broer de robot",
        "Een zomer op Terschelling", "Bouw je eigen game", "De verdwenen schat van kapitein Bos",
        "Alles over bijen", "Het geheim van de oude molen", "Eerste hulp bij heimwee",
        "Draken bestaan niet", "Op zoek naar Mars", "De voetbalschool", "Koken zonder recept",
        "Het meisje dat niet kon stoppen met rennen", "Waarom de zee zout is",
        "Sterren kijken voor beginners", "De ruimtereis van Juul", "Slimme machines",
        "Dieren in de winter", "De draak van de brandweer", "Vriendschap op kamp",
        "Het raadsel van de vuurtoren", "Bakken met kinderen", "Wolven in Europa",
        "De schoolreis naar de bergen",
    ],
    "Korte omschrijving": [
        "Een meisje ontdekt dat de oude draak onder haar school niet dood is. Alleen zij kan hem horen, en elke nacht wordt hij onrustiger. Samen met haar beste vriend zoekt ze uit hoe ze de draak kan tegenhouden voordat de school instort. Maar niet iedereen op school wil dat de draak verdwijnt.",
        "Hoe eten, slapen en douchen astronauten aan boord van het ruimtestation? Dit boek neemt je mee achter de schermen, van de training op aarde tot de terugkeer na een half jaar in de ruimte. Met veel foto's en antwoorden op vragen van kinderen.",
        "Twee broers vinden in het bos een enorm ei dat warm aanvoelt. Als het uitkomt, staat het hele dorp op zijn kop. De jongens moeten het geheim bewaren, want de burgemeester wil alles wat vreemd is laten opruimen. Een spannend verhaal over broers, een geheim en een verrassing die groter is dan ze dachten.",
        "Na de scheiding van haar ouders verhuist Noor naar de stad. Op een nieuwe school kent ze niemand en alles is anders dan thuis. Langzaam leert ze nieuwe vrienden kennen, maar ze mist haar oude dorp en haar vader. Een verhaal over verhuizen, vriendschap en opnieuw beginnen.",
        "Op de schoolkrant lekt een geheim uit. Wie heeft gepraat, en hoe houd je de redactie bij elkaar? Quinten is hoofdredacteur en moet kiezen tussen de waarheid en zijn vrienden. Een verhaal over school, vriendschap en de macht van een krant.",
        "Waarom barsten vulkanen uit en hoe meten onderzoekers dat? In dit boek lees je wat er onder de grond gebeurt en wat je doet als er een vulkaan in de buurt actief wordt. Met verhalen van vulkanologen die dicht bij de lava werken.",
        "In een dorp in de bergen wordt de laatste wolf gezien. Een jongen wil de wolf redden, maar de boeren willen hem weg. Terwijl de winter dichterbij komt, moet de jongen kiezen tussen zijn dorp en het dier. Een verhaal over vriendschap tussen mens en dier.",
        "Sems ouders kopen een robot die alles beter kan dan hij. Op school, thuis en op het voetbalveld lijkt de robot overal beter in. Totdat de robot iets doet wat niemand had verwacht. Een grappig verhaal over broers, techniek en wat een mens bijzonder maakt.",
        "Drie vrienden werken een zomer lang in een strandtent op Terschelling. Ze ontdekken dat het eiland een geheim bewaart, verstopt tussen de duinen. Wat begint als een vakantiebaantje, wordt een zoektocht naar een oude schat.",
        "Stap voor stap een eigen computerspel maken, van idee tot speelbare versie. Je leert programmeren zonder ervaring, ontwerpt je eigen personages en test je spel met vrienden. In dit boek maak je elke week een nieuw level.",
        "Een oude kaart, een verdwenen schip en een klas die op schoolreis per ongeluk een echte schat vindt. Maar de schat is niet van hen, en iemand anders zoekt er al jaren naar. Een spannend verhaal vol raadsels, geheime gangen en piraten.",
        "Hoe werkt een bijenvolk en waarom verdwijnen bijen? In dit boek lees je wat een koningin doet, hoe bijen honing maken en wat je in je eigen tuin kunt doen om ze te helpen. Met proefjes en weetjes.",
        "De oude molen aan de rand van het dorp draait soms 's nachts. Lotte en haar buurjongen gaan op onderzoek uit en vinden een verborgen kamer. Wie woonde hier vroeger, en waarom mocht niemand het weten? Een spannend verhaal over een geheim dat al honderd jaar bewaard wordt.",
        "Tips voor als je voor het eerst van huis bent, op kamp, op schoolreis of in een gastgezin. Wat doe je als je je verdrietig voelt en wie kun je dan om hulp vragen? Een boek vol praktische tips en verhalen van andere kinderen.",
        "Een jongen die niet in draken gelooft, krijgt er een in zijn tuin. En de draak gelooft niet in jongens. Samen komen ze erachter dat ze meer op elkaar lijken dan ze dachten. Een grappig verhaal over vriendschap en over geloven in dingen die je niet kunt zien.",
        "Wat weten we over de rode planeet? In dit boek lees je hoe de missies naar Mars verlopen, wat robots daar ontdekken en of er ooit mensen zullen wonen. Met foto's uit de ruimte en tekeningen van toekomstige steden.",
        "Op de voetbalschool moet Amir kiezen tussen zijn beste vriend en een plek in het eerste team. Op het veld wordt de strijd steeds harder en op school raakt hij zijn vrienden kwijt. Een verhaal over sport, vriendschap en eerlijk spelen.",
        "Koken op gevoel: wat past bij wat, hoe je proeft en hoe je bijstuurt. Met twintig basisgerechten die je steeds anders kunt maken. In dit boek leer je koken zonder dat je alles hoeft af te wegen.",
        "Fenna rent elke dag harder en verder, tot haar lichaam zegt dat het genoeg is geweest. Haar trainer ziet het niet en haar vriendin durft niets te zeggen. Een eerlijk verhaal over sport, druk en het durven stoppen.",
        "Korte antwoorden op grote vragen over de zee: waarom is het water zout, hoe ontstaan getijden en wat leeft er op de bodem? Met proefjes die je thuis kunt doen en weetjes over stromingen.",
        "Hoe vind je de Grote Beer, en wat zie je door een telescoop? In dit boek leer je de sterren en planeten herkennen, ook als je in de stad woont. Met kaarten van de sterrenhemel en tips om zelf de ruimte te bekijken.",
        "Juul wil astronaut worden en mag meedoen aan een training voor kinderen. Maar tijdens de proef gaat er iets mis in het ruimtestation. Een spannend verhaal over durf, samenwerken en dromen van een reis naar de sterren.",
        "Wat kan een robot en wat kan hij nog niet? In dit boek lees je hoe computers leren, hoe robots worden gebouwd en wat dat betekent voor de toekomst. Met opdrachten om zelf een eenvoudige robot te programmeren.",
        "Wat doet een egel als het vriest, en waar gaan vogels naartoe? In dit boek lees je hoe dieren de winter doorkomen, van de winterslaap tot de lange reis naar het zuiden. Met foto's, weetjes en tips om dieren te helpen in je eigen tuin.",
        "In het stadje woont een jonge draak die per ongeluk brandweerman wordt. Hij blust branden met zijn adem, maar kan zelf geen vuur meer maken. Een grappig verhaal over een draak die zijn plek zoekt.",
        "Tijdens het schoolkamp raakt Mila haar beste vriendin kwijt aan de nieuwe leerling. Ze voelt zich buitengesloten en heeft heimwee naar huis. Een verhaal over kamp, vriendschap en jaloezie.",
        "Op een eiland staat een vuurtoren waar al jaren niemand meer woont. Maar elke nacht brandt er licht. Twee vrienden gaan op onderzoek uit en ontdekken een geheim dat het eiland bijna verloor.",
        "Van pannenkoeken tot taart: twintig recepten om zelf te maken, met uitleg stap voor stap. In dit boek leer je hoe je deeg maakt, hoe je proeft en wat je doet als iets mislukt.",
        "Waar leven wolven en waarom keren ze terug naar Nederland? In dit boek lees je hoe wolven leven, hoe ze jagen en hoe mensen en wolven kunnen samenleven. Met verhalen van boswachters en boeren.",
        "Een klas gaat op schoolreis naar de bergen, maar een sneeuwstorm verandert alles. Samen met de juf moet Daan met zijn klasgenoten een nacht overleven in een oude hut. Een spannend verhaal over samenwerken en over wat vriendschap waard is.",
    ],
})

titles = books["Titel"].tolist()
texts = books["Korte omschrijving"].tolist()
books

,Titel,Korte omschrijving
0,De draak onder de school,Een meisje ontdekt dat de oude draak onder haa...
1,Leven in de ruimte,"Hoe eten, slapen en douchen astronauten aan bo..."
2,Het drakenei,Twee broers vinden in het bos een enorm ei dat...
3,Nieuw in de stad,Na de scheiding van haar ouders verhuist Noor ...
4,Code rood op de redactie,Op de schoolkrant lekt een geheim uit. Wie hee...
5,Vulkanen van dichtbij,Waarom barsten vulkanen uit en hoe meten onder...
6,De laatste wolf,In een dorp in de bergen wordt de laatste wolf...
7,Mijn broer de robot,Sems ouders kopen een robot die alles beter ka...
8,Een zomer op Terschelling,Drie vrienden werken een zomer lang in een str...
9,Bouw je eigen game,"Stap voor stap een eigen computerspel maken, v..."


## 3. Opschonen en tokeniseren

Drie stappen, in een functie die we straks ook in `recommender.py` gebruiken:

1. **Kleine letters**: "Draak" aan het begin van een zin en "draak" in het midden zijn hetzelfde woord.
2. **Leestekens weg**: "draak." en "draak," worden "draak".
3. **Tokeniseren**: de tekst in losse woorden knippen.

`re.findall(r"\w+", ...)` doet 2 en 3 in één keer: het pakt alle reeksen letters en cijfers en laat de rest liggen. Een aparte tokenizer-bibliotheek is hier niet nodig.

In [103]:
# Basis-tokenizer: alleen kleine letters en losse woorden, verder niets.
# Deze moet nog uitgebreid worden met stopwoorden (sectie 5) en eventueel stemming (sectie 6); dat komt later.
def clean(text: str) -> str:
    return text.lower()


def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", clean(text))


for sentence in demo_sentences:
    print(f"{sentence!r:40} -> {tokenize(sentence)}")

print()
print(texts[0])
print(tokenize(texts[0]))

'De hond bijt de man'                    -> ['de', 'hond', 'bijt', 'de', 'man']
'De kat slaapt'                          -> ['de', 'kat', 'slaapt']
'De hond slaapt, de kat slaapt'          -> ['de', 'hond', 'slaapt', 'de', 'kat', 'slaapt']

Een meisje ontdekt dat de oude draak onder haar school niet dood is. Alleen zij kan hem horen, en elke nacht wordt hij onrustiger. Samen met haar beste vriend zoekt ze uit hoe ze de draak kan tegenhouden voordat de school instort. Maar niet iedereen op school wil dat de draak verdwijnt.
['een', 'meisje', 'ontdekt', 'dat', 'de', 'oude', 'draak', 'onder', 'haar', 'school', 'niet', 'dood', 'is', 'alleen', 'zij', 'kan', 'hem', 'horen', 'en', 'elke', 'nacht', 'wordt', 'hij', 'onrustiger', 'samen', 'met', 'haar', 'beste', 'vriend', 'zoekt', 'ze', 'uit', 'hoe', 'ze', 'de', 'draak', 'kan', 'tegenhouden', 'voordat', 'de', 'school', 'instort', 'maar', 'niet', 'iedereen', 'op', 'school', 'wil', 'dat', 'de', 'draak', 'verdwijnt']


## 4. Bag of words met `CountVectorizer`

Bag of words in het kort: een **vocabulaire** (elk woord één vaste plek), per document een **telling**, en die telling op de vaste plekken in een rij zetten. Het vocabulaire wordt opgebouwd op *alle* documenten samen, niet per document; anders zijn de rijen niet vergelijkbaar. scikit-learn doet dit voor je. Vergelijk de uitkomst met sheet 7: `hond, kat, bijt, man, slaapt`. Hier staat ook `de` in het vocabulaire; op de sheet was dat weggelaten.

Let op: **woordvolgorde verdwijnt.** "De hond bijt de man" en "De man bijt de hond" krijgen dezelfde vector. Vandaar de naam: een zak met woorden.

Twee dingen om te onthouden:

- `fit` leert het vocabulaire op alle documenten; `transform` maakt de matrix. **Eén keer `fit`, daarna alleen `transform`.** Opnieuw `fit` op nieuwe teksten geeft een ander vocabulaire, en dan kun je vectoren niet meer vergelijken.
- De matrix is **sparse**: alleen de niet-nullen worden bewaard. Bij 200 titels merk je er niets van; bij 50.000 documenten past `.toarray()` niet meer in het geheugen. Hieronder mag het, want het is klein.

De standaard-tokenizer van scikit-learn laat woorden van één letter weg (`token_pattern`). We geven onze eigen `tokenize` mee.

In [104]:
from sklearn.feature_extraction.text import CountVectorizer

# We geven de basis-tokenizer mee; die wordt later uitgebreid (stopwoorden, stemming).
cv = CountVectorizer(tokenizer=tokenize, token_pattern=None, lowercase=False)
X_sentences = cv.fit_transform(demo_sentences)

print("vocabulaire:", cv.get_feature_names_out())
print("BoW Matrix:\n", X_sentences.toarray())


vocabulaire: ['bijt' 'de' 'hond' 'kat' 'man' 'slaapt']
BoW Matrix:
 [[1 2 1 0 1 0]
 [0 1 0 1 0 1]
 [0 2 1 1 0 2]]


Om het wat makkelijker te lezen inclusief de titel en de woorden:

In [105]:
pd.DataFrame(X_sentences.toarray(), columns=cv.get_feature_names_out(), index=demo_sentences).head(10)

,bijt,de,hond,kat,man,slaapt
De hond bijt de man,1,2,1,0,1,0
De kat slaapt,0,1,0,1,0,1
"De hond slaapt, de kat slaapt",0,2,1,1,0,2


**Wat is `X_sentences` eigenlijk?** Het resultaat van `fit_transform` is geen `DataFrame` en geen gewone lijst, maar een **sparse matrix** uit de bibliotheek SciPy (het type is `csr_matrix`). Zo'n matrix bewaart alleen de cellen die niet nul zijn. Dat scheelt veel geheugen: in een tekstmatrix is de meeste ruimte nul, want een omschrijving gebruikt maar een klein deel van alle woorden in het vocabulaire. Wat je eruit kunt opvragen:

| Eigenschap | Betekenis |
|---|---|
| `X.shape` | De vorm als tuple `(aantal rijen, aantal kolommen)`. |
| `X.shape[0]` | Het aantal rijen: het aantal **documenten** (hier zinnen of omschrijvingen). |
| `X.shape[1]` | Het aantal kolommen: het aantal **woorden in het vocabulaire**. |
| `X.nnz` | "number of non-zeros": hoeveel cellen **niet nul** zijn, dus hoeveel getallen er echt zijn opgeslagen. |
| `X.shape[0] * X.shape[1]` | Het totale aantal cellen als de matrix volledig was uitgeschreven. |
| `X.toarray()` | Zet de matrix om naar een gewone `numpy`-array, met alle nullen erbij. Handig om te bekijken, maar bij een grote matrix past dat niet in het geheugen. |

Vergelijk `nnz` met het totaal aantal cellen: dat zegt hoe leeg de matrix is.

In [106]:
print("type:", type(X_sentences))
print("vorm (documenten x woorden):", X_sentences.shape)
print("aantal documenten:", X_sentences.shape[0])
print("aantal woorden in het vocabulaire:", X_sentences.shape[1])
print("niet-nullen (nnz):", X_sentences.nnz, "van", X_sentences.shape[0] * X_sentences.shape[1], "cellen")

type: <class 'scipy.sparse._csr.csr_matrix'>
vorm (documenten x woorden): (3, 6)
aantal documenten: 3
aantal woorden in het vocabulaire: 6
niet-nullen (nnz): 11 van 18 cellen


Nu op de omschrijvingen. Kijk naar de vorm van de matrix en naar de kolomsom: welke woorden komen het vaakst voor?

In [107]:
cv = CountVectorizer(tokenizer=tokenize, token_pattern=None, lowercase=False)
X_books = cv.fit_transform(texts)
print("vorm van de matrix (documenten x woorden):", X_books.shape)
print("opgeslagen getallen (niet-nullen):", X_books.nnz, "van", X_books.shape[0] * X_books.shape[1])

# Hoe vaak komt elk woord voor, over alle documenten samen?
# X_books.sum(axis=0) telt per kolom (= per woord) alle rijen (= documenten) bij elkaar op.
# Het resultaat is een matrix met één rij, bijvoorbeeld [[63 60 53 ...]]. Met .A1 maak je daar een
# gewone, platte rij getallen van (zonder die extra "rij"), zodat pandas er een Series van kan maken.
# De index van die Series zijn de woorden, in dezelfde volgorde als de kolommen.
freq = pd.Series(X_books.sum(axis=0).A1, index=cv.get_feature_names_out())
freq.sort_values(ascending=False).head(25)

vorm van de matrix (documenten x woorden): (30, 453)
opgeslagen getallen (niet-nullen): 1002 van 13590


een        63
en         60
de         53
je         36
in         29
op         23
het        23
wat        21
hoe        19
over       18
van        17
met        16
ze         14
verhaal    14
boek       12
dit        11
dat        11
die        10
haar       10
zijn        9
is          9
naar        9
er          8
als         8
niet        8
dtype: int64

**Wat zie je?** De top van de lijst bestaat uit "een", "en", "de", "je", "in", "op" en "het". Die staan in bijna elke omschrijving en zeggen niets over het onderwerp. Voor een aanbeveling willen we juist de woorden die een boek *onderscheiden*.

## 5. Stopwoorden

Een stopwoordenlijst is een grove oplossing: iemand heeft bepaald welke woorden je weggooit. Bekijk de lijst daarom altijd, en vul hem aan met woorden die *in deze catalogus* niets zeggen. In sectie 8 (TF-IDF) zie je dat IDF het grootste deel van dit werk vanzelf doet.

scikit-learn heeft alleen een Engelse lijst; voor Nederlands gebruiken we die van NLTK.

In [108]:
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)          # eenmalig, daarna uit de lokale cache

stop_words = set(stopwords.words("dutch"))
print("NLTK Nederlandse stopwoorden:", len(stop_words))
print(sorted(stop_words)[:40], "...")

NLTK Nederlandse stopwoorden: 101
['aan', 'al', 'alles', 'als', 'altijd', 'andere', 'ben', 'bij', 'daar', 'dan', 'dat', 'de', 'der', 'deze', 'die', 'dit', 'doch', 'doen', 'door', 'dus', 'een', 'eens', 'en', 'er', 'ge', 'geen', 'geweest', 'haar', 'had', 'heb', 'hebben', 'heeft', 'hem', 'het', 'hier', 'hij', 'hoe', 'hun', 'iemand', 'iets'] ...


In [109]:
# Eerste uitbreiding van de tokenizer: de standaard stopwoorden weglaten.
def tokenize_stop(text: str) -> list[str]:
    return [w for w in tokenize(text) if w not in stop_words]


cv_stop = CountVectorizer(tokenizer=tokenize_stop, token_pattern=None, lowercase=False)
X_stop = cv_stop.fit_transform(texts)
print("vocabulaire zonder stopwoorden:", X_stop.shape[1], "woorden (was", X_books.shape[1], ")")

# Woordfrequenties, net als hierboven (.A1: zie de uitleg bij `freq`).
freq_stop = pd.Series(X_stop.sum(axis=0).A1, index=cv_stop.get_feature_names_out())
freq_stop.sort_values(ascending=False).head(15)

vocabulaire zonder stopwoorden: 386 woorden (was 453 )


verhaal        14
boek           12
vriendschap     7
school          7
vrienden        6
draak           6
lees            6
oude            6
geheim          6
waarom          5
tussen          5
doet            5
dorp            5
robot           5
spannend        5
dtype: int64

**Wat zie je?** De meeste woorden bovenaan zijn nu inhoud. Maar "verhaal" en "boek" staan er nog steeds in: die komen in een catalogus bijna overal voor en onderscheiden geen enkel boek. Kijk ook naar wat er nét onder staat: woorden als "oude", "geheim" of "dorp" komen in veel omschrijvingen voor en zeggen weinig over één specifieke titel. Daar is een lijst te grof voor; dat lost IDF op.

**Opdracht: vul de lijst zelf aan.** Een standaardlijst kent jullie catalogus niet. Kijk naar de top 15 hierboven: welke woorden komen vaak voor maar onderscheiden geen enkel boek? Leg in jullie notebook kort uit wat je toevoegt en waarom.

In [110]:
# Startpunt: vul aan met woorden die in jullie catalogus overal in staan. Dit is een keuze; noteer hem.
CUSTOM_STOPWORDS = {"boek", "verhaal", "jongen", "meisje", "jaar"}
all_stop_words = stop_words | CUSTOM_STOPWORDS


# Dezelfde tokenizer, nu met de standaardlijst plus jullie eigen aanvulling.
# Deze versie gebruiken we in de rest van de notebook.
def tokenize_nostop(text: str) -> list[str]:
    return [w for w in tokenize(text) if w not in all_stop_words]


cv_custom = CountVectorizer(tokenizer=tokenize_nostop, token_pattern=None, lowercase=False)
X_custom = cv_custom.fit_transform(texts)
print("vocabulaire met eigen aanvulling:", X_custom.shape[1], "woorden (standaardlijst:", X_stop.shape[1], ")")

freq_custom = pd.Series(X_custom.sum(axis=0).A1, index=cv_custom.get_feature_names_out())
freq_custom.sort_values(ascending=False).head(15)

vocabulaire met eigen aanvulling: 384 woorden (standaardlijst: 386 )


vriendschap    7
school         7
vrienden       6
lees           6
draak          6
geheim         6
oude           6
tussen         5
robot          5
spannend       5
dorp           5
maken          5
waarom         5
doet           5
tips           4
dtype: int64

## 6. Optioneel: stemming

"draak" en "draken" zijn voor de teller twee kolommen. Een **stemmer** knipt uitgangen af met regels ("draken" → "drak"); **lemmatisering** zoekt de woordenboekvorm op ("draken" → "draak") maar heeft een taalmodel nodig (voor Nederlands bijvoorbeeld spaCy). In deze les volstaat de stemmer; of je hem gebruikt is een keuze die je kort toelicht in je notebook.

Let op wat het *niet* oplost: "drakenei" blijft een eigen woord, en "bende" en "bandieten" blijven twee kolommen.

In [111]:
from nltk.stem.snowball import SnowballStemmer

stemmer = SnowballStemmer("dutch")
stem = stemmer.stem

for w in ["draak", "draken", "drakenei", "lopen", "liep", "scholen", "school"]:
    print(f"{w:10} -> {stem(w)}")


# Tweede uitbreiding van de tokenizer: elk woord terugbrengen tot zijn stam.
def tokenize_stemmed(text: str) -> list[str]:
    return [stem(w) for w in tokenize(text)]


print()
print(tokenize_stemmed(texts[0]))

draak      -> drak
draken     -> drak
drakenei   -> drakenei
lopen      -> lop
liep       -> liep
scholen    -> schol
school     -> schol

['een', 'meisj', 'ontdekt', 'dat', 'de', 'oud', 'drak', 'onder', 'har', 'schol', 'niet', 'dod', 'is', 'allen', 'zij', 'kan', 'hem', 'hor', 'en', 'elk', 'nacht', 'wordt', 'hij', 'onrustiger', 'sam', 'met', 'har', 'best', 'vriend', 'zoekt', 'ze', 'uit', 'hoe', 'ze', 'de', 'drak', 'kan', 'tegenhoud', 'voordat', 'de', 'schol', 'instort', 'mar', 'niet', 'iederen', 'op', 'schol', 'wil', 'dat', 'de', 'drak', 'verdwijnt']


In [112]:
# Doet tokenize_stemmed iets? Vergelijk met de gewone tokenizer.
example = "De draken wonen in drakenholen; de draak slaapt, de scholen sluiten."
print("zonder stemming:", tokenize(example))
print("met stemming:   ", tokenize_stemmed(example))


# Stopwoorden en stemming samen: eerst de stopwoorden eruit, dan elk woord naar zijn stam.
def tokenize_stemmed_nostop(text: str) -> list[str]:
    return [stem(w) for w in tokenize_nostop(text)]


# Eerlijke vergelijking: beide vocabulaires hebben de stopwoorden al weggelaten.
cv_stem = CountVectorizer(tokenizer=tokenize_stemmed_nostop, token_pattern=None, lowercase=False)
X_stem = cv_stem.fit_transform(texts)
print()
print("vocabulaire zonder stemming:", X_custom.shape[1], "woorden")
print("vocabulaire met stemming:   ", X_stem.shape[1], "woorden")

# Welke woorden zijn samengevallen? Groepeer de woorden uit de omschrijvingen op hun stam.
groups = {}
for w in cv_custom.get_feature_names_out():
    groups.setdefault(stem(w), []).append(w)
{s: ws for s, ws in groups.items() if len(ws) > 1}

zonder stemming: ['de', 'draken', 'wonen', 'in', 'drakenholen', 'de', 'draak', 'slaapt', 'de', 'scholen', 'sluiten']
met stemming:    ['de', 'drak', 'won', 'in', 'drakenhol', 'de', 'drak', 'slaapt', 'de', 'schol', 'sluit']

vocabulaire zonder stemming: 384 woorden
vocabulaire met stemming:    361 woorden


{'astronaut': ['astronaut', 'astronauten'],
 'dier': ['dier', 'dieren'],
 'drak': ['draak', 'draken'],
 'eerst': ['eerst', 'eerste'],
 'geheim': ['geheim', 'geheime'],
 'jar': ['jaar', 'jaren'],
 'jong': ['jonge', 'jongen'],
 'kaart': ['kaart', 'kaarten'],
 'kop': ['kop', 'kopen'],
 'lang': ['lang', 'lange'],
 'ler': ['leer', 'leren'],
 'mak': ['maak', 'maken'],
 'nacht': ['nacht', 'nachts'],
 'nieuw': ['nieuw', 'nieuwe'],
 'planet': ['planeet', 'planeten'],
 'robot': ['robot', 'robots'],
 'spel': ['spel', 'spelen'],
 'sted': ['steden', 'steeds'],
 'toekomst': ['toekomst', 'toekomstige'],
 'vind': ['vind', 'vinden'],
 'vriend': ['vriend', 'vrienden'],
 'vulkan': ['vulkaan', 'vulkanen'],
 'weg': ['weg', 'wegen']}

---

## 7. Opdracht: bag of words op jullie eigen dataset

Tot hier hebben jullie bag of words gedaan op de voorbeeldboeken. Doe dit nu op jullie **eigen dataset**, en maak bewuste keuzes over wat je met de tekst doet. Voordat je verdergaat met TF-IDF moet dit werken.

**Minimale keuzes (verplicht).** Twee keuzes moet je maken, bekijken en onderbouwen:

| # | Keuze | Wat je doet | Minimaal |
|---|---|---|---|
| 1 | Stopwoorden | Vul `CUSTOM_STOPWORDS` (sectie 5) aan met woorden uit de top 25 van **jullie** data die niets zeggen over een titel. | 5 woorden |
| 2 | Stemming | Kies of jullie `tokenize_stemmed_nostop` (sectie 6) gebruiken: probeer het met en zonder en bekijk welke woorden samenvallen. | 1 keuze, met reden |

Leg bij elke keuze kort uit in jullie notebook: wat je veranderde, waarom, wat het deed met het vocabulaire en of je het houdt.

## 8. TF-IDF

Een telling zegt hoe vaak een woord voorkomt, maar niet of het woord iets zegt. TF-IDF weegt dat mee:

- **TF** (term frequency): hoe vaak staat het woord in dit document? Hier gewoon de telling.
- **IDF** (inverse document frequency): in hoe weinig documenten staat het woord? `log(N / df)`, met N het aantal documenten en df het aantal documenten waar het woord in staat. Een woord dat overal in staat krijgt `log(1) = 0` en telt niet meer mee: de computer vond zelf een stopwoord.
- **TF-IDF** = TF × IDF.

scikit-learn doet het net iets anders: `ln((1 + N) / (1 + df)) + 1`, en daarna wordt elke rij gedeeld door haar lengte (L2-normalisatie). Daardoor wijken de getallen af van de formule hierboven; de volgorde van de woorden niet.

## 9. TF-IDF op de omschrijvingen

Dezelfde vectorizer op de omschrijvingen. In de eerste cel hieronder kies je met `chosen_tokenizer` welke tokenizer je gebruikt: met of zonder stemming, met of zonder stopwoorden. 

In [113]:
from sklearn.feature_extraction.text import TfidfVectorizer

chosen_tokenizer = tokenize_nostop # kies je eigen tokenizer, bijvoorbeeld met of zonder stemming, met of zonder stopwoorden

tfidf = TfidfVectorizer(tokenizer=chosen_tokenizer, token_pattern=None, lowercase=False, min_df=1)
X_tfidf = tfidf.fit_transform(books["Korte omschrijving"])
words = tfidf.get_feature_names_out()
print("matrix:", X_tfidf.shape)

print("Matrix (TF-IDF):\n", X_tfidf.toarray())



matrix: (30, 384)
Matrix (TF-IDF):
 [[0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.23309459 0.23309459 ... 0.         0.         0.        ]
 [0.19529432 0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]


Omzetten naar een leesbare matrix:

In [114]:
tfidf_dataframe = pd.DataFrame(X_tfidf.toarray(), columns=words, index=titles)

tfidf_dataframe.head(10)

,aanvoelt,aarde,achter,actief,adem,af,alleen,amir,anders,antwoorden,astronaut,astronauten,barsten,basisgerechten,beer,...,wonen,woonde,woont,zee,zeggen,zegt,zie,zien,ziet,zoekt,zoektocht,zomer,zout,zuiden,zullen
De draak onder de school,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.196395,0.0,0.000000,0.00000,0.0,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.160005,0.000000,0.000000,0.0,0.0,0.0
Leven in de ruimte,0.000000,0.233095,0.233095,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.20783,0.0,0.233095,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0
Het drakenei,0.195294,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.00000,0.0,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0
Nieuw in de stad,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.178584,0.00000,0.0,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0
Code rood op de redactie,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.00000,0.0,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0
Vulkanen van dichtbij,0.000000,0.000000,0.000000,0.259074,0.0,0.0,0.000000,0.0,0.000000,0.00000,0.0,0.000000,0.259074,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0
De laatste wolf,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.00000,0.0,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0
Mijn broer de robot,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.00000,0.0,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0
Een zomer op Terschelling,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.00000,0.0,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.249981,0.249981,0.0,0.0,0.0
Bouw je eigen game,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.00000,0.0,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0


In [115]:
# Per omschrijving de drie woorden met het hoogste TF-IDF-gewicht, voor de eerste 6 omschrijvingen
# (of minder, als er minder omschrijvingen zijn).
for i in range(min(6, len(books))):
    # Rij i van de matrix: één getal per woord uit het vocabulaire.
    # .toarray() maakt er een gewone array van met vorm (1, aantal woorden), een "tabel" met één rij.
    # .ravel() maakt daar een platte lijst van (aantal woorden,), zodat je er met één index in kunt.
    row = X_tfidf[i].toarray().ravel()

    # row.argsort() geeft de POSITIES van de woorden, gesorteerd van laagste naar hoogste gewicht
    # (het geeft dus niet de gewichten zelf, maar waar ze staan).
    # [::-1] draait die volgorde om, zodat het hoogste gewicht vooraan staat.
    # [:3] pakt daarvan de eerste drie: de posities van de drie zwaarste woorden.
    # words[j] zoekt bij elke positie j het bijbehorende woord op.
    top = [words[j] for j in row.argsort()[::-1][:3]]
    print(f"{titles[i]:40} {top}")

De draak onder de school                 ['draak', 'school', 'tegenhouden']
Leven in de ruimte                       ['neemt', 'slapen', 'terugkeer']
Het drakenei                             ['broers', 'geheim', 'uitkomt']
Nieuw in de stad                         ['nieuwe', 'verhuizen', 'verhuist']
Code rood op de redactie                 ['redactie', 'schoolkrant', 'quinten']
Vulkanen van dichtbij                    ['buurt', 'dicht', 'gebeurt']


In [116]:
# Waarom deze drie? Bekijk per woord het gewicht in de omschrijving en in hoeveel omschrijvingen het woord staat (df).
# df: (X_tfidf > 0) markeert waar een woord voorkomt; .sum(axis=0).A1 telt per woord de documenten (zie `freq` in sectie 4).
doc_freq = (X_tfidf > 0).sum(axis=0).A1

for i in [0, 1]:
    row = X_tfidf[i].toarray().ravel()

    print(titles[i])
    table = pd.DataFrame({"woord": words, "gewicht": row.round(3), "df": doc_freq}) \
              .query("gewicht > 0").sort_values("gewicht", ascending=False).head(8).reset_index(drop=True)
    print(table)
    print()

De draak onder de school
      woord  gewicht  df
0     draak    0.480   3
1    school    0.416   5
2      dood    0.196   1
3    alleen    0.196   1
4   voordat    0.196   1
5     horen    0.196   1
6   instort    0.196   1
7  iedereen    0.196   1

Leven in de ruimte
         woord  gewicht  df
0        aarde    0.233   1
1       achter    0.233   1
2  astronauten    0.233   1
3        boord    0.233   1
4      douchen    0.233   1
5          mee    0.233   1
6         eten    0.233   1
7         half    0.233   1



**Wat zie je?** Bij "De draak onder de school" werkt het zoals bedoeld: "draak" en "school" staan bovenaan, omdat ze vaak in deze omschrijving voorkomen en niet in elke omschrijving staan. Kijk daarna naar de rest: alle woorden die in maar één omschrijving staan (df = 1) hebben precies hetzelfde gewicht. Bij "Leven in de ruimte" geldt dat voor bijna alle woorden, en "ruimte" zelf staat niet eens in de eerste acht.

Met 30 omschrijvingen kan TF-IDF dus nog niet zien welk van die zeldzame woorden over het onderwerp gaat. De volgorde binnen zo'n groep is toevallig. Dat wordt beter met een grotere catalogus, en met `min_df` (hieronder) haal je de toevallige woorden weg.

**Knop om aan te draaien:** `min_df=2` gooit woorden weg die in maar één omschrijving staan. Dat maakt het vocabulaire veel kleiner, maar bij korte omschrijvingen verdwijnen juist de onderscheidende woorden. Vergelijk hieronder de grootte van het vocabulaire met `min_df=1` en `min_df=2`.

In [117]:
tfidf_min2 = TfidfVectorizer(tokenizer=chosen_tokenizer, token_pattern=None, lowercase=False, min_df=2)
X_tfidf_min2 = tfidf_min2.fit_transform(books["Korte omschrijving"])
print("vocabulaire met min_df=1:", X_tfidf.shape[1], "  met min_df=2:", X_tfidf_min2.shape[1])

vocabulaire met min_df=1: 384   met min_df=2: 98


## 10. Cosine-gelijkenis en de top 5

Elke omschrijving is nu een vector / pijl. **Cosine** meet de hoek tussen twee pijlen: 1 is dezelfde richting, 0 is loodrecht (geen woord gemeen). De lengte van de tekst doet er niet toe; daarom gebruiken we geen Euclidische afstand.

Omdat `TfidfVectorizer` elke rij al op lengte 1 zet, is cosine hier gewoon het inproduct. `cosine_similarity` geeft meteen de hele matrix: documenten × documenten.

In [118]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(X_tfidf)
print("gelijkenismatrix:", sim.shape)
pd.DataFrame(sim[:6, :6], index=titles[:6], columns=[t[:14] for t in titles[:6]]).round(2)

gelijkenismatrix: (30, 30)


,De draak onder,Leven in de ru,Het drakenei,Nieuw in de st,Code rood op d,Vulkanen van d
De draak onder de school,1.00,0.0,0.00,0.08,0.08,0.0
Leven in de ruimte,0.00,1.0,0.00,0.00,0.00,0.0
Het drakenei,0.00,0.0,1.00,0.02,0.05,0.0
Nieuw in de stad,0.08,0.0,0.02,1.00,0.08,0.0
Code rood op de redactie,0.08,0.0,0.05,0.08,1.00,0.0
Vulkanen van dichtbij,0.00,0.0,0.00,0.00,0.00,1.0


De diagonaal is 1 (elke tekst lijkt perfect op zichzelf). Voor een top 5 sla je die over.

`top5` is het eerste stukje aanbeveling: *"omdat je X las"*.

In [119]:
def top5(title: str, n: int = 5) -> pd.DataFrame:
    i = titles.index(title)
    scores = sim[i].copy()
    scores[i] = -1                                  # zichzelf overslaan
    best = scores.argsort()[::-1][:n]
    return pd.DataFrame({"titel": [titles[j] for j in best], "cosine": scores[best].round(3)})


print("Omdat je las:", titles[0])
top5(titles[0])

Omdat je las: De draak onder de school


,titel,cosine
0,De draak van de brandweer,0.239
1,De voetbalschool,0.142
2,Draken bestaan niet,0.120
3,De schoolreis naar de bergen,0.091
4,Nieuw in de stad,0.083


## 11. Wat TF-IDF niet ziet

Kijk naar de top 5 van "De draak onder de school". Staat "Het drakenei" erbij? Nee: "draak" en "drakenei" zijn voor TF-IDF twee totaal verschillende kolommen, dus die twee omschrijvingen hebben geen enkel woord gemeen. En "Nieuw in de stad" of "De voetbalschool" staan er wel in, vooral doordat ze het woord "school" gemeen hebben met "De draak onder de school", terwijl ze verder nergens over draken gaan.

Dat is de blinde vlek van deze les: TF-IDF telt woorden, geen betekenis. Stemming helpt een beetje: in de cel hieronder zie je dat "Draken bestaan niet" hoger scoort doordat "draak" en "draken" dezelfde stam krijgen. Synoniemen lost het niet op. Daarvoor zijn embeddings.

In [ ]:
# Probeer: wat verandert er met stemming?
# We vergelijken expliciet zonder en met stemming, los van `chosen_tokenizer`.
tfidf_plain = TfidfVectorizer(tokenizer=tokenize_nostop, token_pattern=None, lowercase=False)
X_tfidf_plain = tfidf_plain.fit_transform(books["Korte omschrijving"])

tfidf_stem = TfidfVectorizer(tokenizer=tokenize_stemmed_nostop, token_pattern=None, lowercase=False)
X_tfidf_stem = tfidf_stem.fit_transform(books["Korte omschrijving"])

sim_plain = cosine_similarity(X_tfidf_plain)
sim_stem = cosine_similarity(X_tfidf_stem)
i = titles.index(titles[0])
print(f"Vergelijking voor '{titles[i]}':")
comparison = pd.DataFrame({
    "titel": titles,
    "cosine zonder stemming": sim_plain[i].round(3),
    "cosine met stemming": sim_stem[i].round(3),
}).drop(index=i).sort_values("cosine zonder stemming", ascending=False).head(6)
comparison

Vergelijking voor 'De draak onder de school':


,titel,cosine zonder stemming,cosine met stemming
24,De draak van de brandweer,0.239,0.245
16,De voetbalschool,0.142,0.145
14,Draken bestaan niet,0.120,0.208
29,De schoolreis naar de bergen,0.091,0.088
3,Nieuw in de stad,0.083,0.103
4,Code rood op de redactie,0.079,0.101


: 

## Opdracht voor les 2 (af voor de volgende les)

*Doel.* Na deze opdracht maak je van elke omschrijving in jullie catalogus een vector en vraag je voor elke titel de vijf meest gelijkende titels op. En je kunt uitleggen waarom die vijf er staan.

*Stappen.*

- [ ] Gebruik jullie eigen dataset en gebruik **alle** relevante tekstkolommen. Regel dit zelf in jullie code, bijvoorbeeld door kolommen samen te voegen tot één tekst per boek. Let op: de cellen gebruiken `texts` (sectie 4 t/m 6) of `books["Korte omschrijving"]` (sectie 9 t/m 11); pas ze aan naar jullie eigen data.
- [ ] Voeg de TF-IDF functionaliteit toe.
- [ ] Maak `top5` **robuust**. Een goede aanbevelingsfunctie crasht niet als iemand een titel verkeerd typt. Zorg daarom voor het volgende:
  - **Een nette melding bij een onbekende titel.** Staat de titel niet in de catalogus, geef dan een duidelijke, vriendelijke melding terug (bijvoorbeeld: *"De titel 'xyz' staat niet in de catalogus."*) in plaats van een foutmelding.
  - **De ingevoerde titel wordt eerst schoongemaakt.** Maak de titel op dezelfde manier schoon als de titels in de catalogus (bijvoorbeeld met `clean`: kleine letters en overbodige spaties weghalen), en vergelijk de schone invoer met de schone catalogustitels. Zo vindt `top5` ook "  de DRAAK onder de school " nog.
- [ ] Voeg eventueel extra stopwoorden toe aan de tokenizer indien dat nodig is op basis van de belangrijkheid van woorden
- [ ] Vergelijk voor drie vaste titels de top 5 vóór en na elke wijziging en noteer de uitkomsten.
- [ ] Schrijf drie gevallen uit: één waar BoW misgaat door woorden als "de" of "boek", één waar TF-IDF het duidelijk beter doet, en één waar TF-IDF nog steeds misgaat (synoniemen, spelling, samenstellingen).

*Minimale keuzes (verplicht).* Kies drie opties uit onderstaande tabel en vergelijk elk met de standaardinstelling. Verander steeds één ding tegelijk.

| Optie | Waar | Wat het doet | Let op |
|---|---|---|---|
| `min_df` | `TfidfVectorizer(min_df=2)` | Laat woorden weg die in minder dan `min_df` documenten staan. Een getal is een aantal documenten, een kommagetal (bijvoorbeeld `0.01`) een aandeel. Het vocabulaire wordt kleiner en typefouten en zeldzame woorden verdwijnen. | Bij korte omschrijvingen verdwijnen juist de onderscheidende woorden. In de demo ging het van 385 naar 99 woorden. |
| `ngram_range` | `TfidfVectorizer(ngram_range=(1, 2))` | Telt behalve losse woorden ook woordparen mee, zoals "oude molen". Zo blijft een stukje woordvolgorde bewaard. | Het vocabulaire groeit sterk, en de meeste woordparen komen maar één keer voor. Combineer het met `min_df`. |
| `max_df` | `TfidfVectorizer(max_df=0.8)` | Laat woorden weg die in meer dan dit aandeel van de documenten staan. Werkt als een automatische stopwoordenlijst die bij jullie catalogus past. | Een te lage waarde gooit ook woorden weg die wel bij een groot deel van de catalogus horen. |
| `max_features` | `TfidfVectorizer(max_features=500)` | Houdt alleen de 500 meest voorkomende woorden over. | Zonder `min_df` blijven de voorkomende woorden over en niet de onderscheidende. |
| Lemmatisering | bijvoorbeeld spaCy, in een eigen tokenizer | Zoekt de woordenboekvorm op ("draken" wordt "draak"). Preciezer dan stemming. | Heeft een taalmodel nodig en is trager. |

*Testen.* Je kunt op de volgende manier testen:

1. **Meten.** Kijk naar de grootte van het vocabulaire (`X_tfidf.shape[1]`).
2. **Vergelijken.** Bewaar eerst de top 5 van de standaardinstelling voor de drie titels, bijvoorbeeld `top5_before = top5(titles[0])`. Pas daarna de regel `tfidf = TfidfVectorizer(...)` in sectie 9 aan en bouw de matrix en de gelijkenis opnieuw: `X_tfidf = tfidf.fit_transform(books["Korte omschrijving"])` en `sim = cosine_similarity(X_tfidf)`. Vergelijk dan `top5(titles[0])` met `top5_before`; zodra je `sim` opnieuw berekent, is het oude resultaat anders weg.
3. **Robuustheid.** Roep `top5` aan met een titel die niet bestaat, en met een bestaande titel in andere hoofdletters of met extra spaties. De eerste moet een nette melding geven, de tweede moet gewoon de top 5 laten zien.

Zet bij elke wijziging in jullie notebook een korte uitleg: wat je veranderde, waarom, en wat je zag, ook als je hem terugdraait. De uitgebreidere beschrijving van jullie keuzes komt in het verslag.

*Vragen.* Kies een titel die jullie allebei kennen: klopt de top 5? Welk woord veroorzaakt de hoogste gelijkenis (kijk naar de zwaarste TF-IDF-woorden van beide titels)? Vind één paar titels dat volgens jullie bij elkaar hoort maar dat TF-IDF niet bij elkaar zet. Waarom niet, en wat zou daar wel bij helpen?

*Klaar als.* Jullie leveren een werkend notebook in waarin `top5` werkt op elke titel uit jullie csv en een nette melding geeft bij een onbekende titel, met de drie uitgeschreven gevallen en een korte uitleg van jullie keuzes. De uitgebreidere beschrijving staat in het verslag.